In [1]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Advanced Short-Answer Scoring with TF–IDF + Thai BERT embeddings + XGBoost
Author: Your Name

Description:
  This code demonstrates a more advanced approach to short-answer scoring. We:
    1. Read train/test CSVs + sample_submission.csv
    2. Combine question + answer text
    3. Generate two sets of features: TF–IDF vectors and Thai BERT embeddings
    4. Concatenate them to form a unified feature representation
    5. Train an XGBoost regressor
    6. Evaluate via cross-validation
    7. Output predictions to submission.csv

Dependencies (install with pip):
    scikit-learn, xgboost, transformers, pythainlp, tqdm
Usage:
    python advanced_shortanswer.py
"""

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModel  # For Thai BERT

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import xgboost as xgb


###############################################################################
# 1) Reading Data
###############################################################################
def read_data():
    """
    Reads train.csv, test.csv, and sample_submission.csv from the current directory (./).
    Expects columns:
      train.csv => [ID, set, question, answer, score]
      test.csv  => [ID, set, question, answer]
      sample_submission.csv => [ID, score]
    Returns:
      train_df, test_df, sub_df
    """
    train_path = os.path.join(REPO_PATH, "data", "train.csv")
    test_path = os.path.join(REPO_PATH, "data", "test.csv")
    sample_sub_path = os.path.join(REPO_PATH, "data", "sample_submission.csv")

    for p in [train_path, test_path, sample_sub_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing file: {p}. Please check your paths.")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    sub_df = pd.read_csv(sample_sub_path)

    return train_df, test_df, sub_df


###############################################################################
# 2) Text Combination
###############################################################################
def combine_text(question, answer):
    """
    Join question + answer into one string.
    You might add advanced tokenization or cleaning here for Thai text.
    """
    q = question if isinstance(question, str) else ""
    a = answer if isinstance(answer, str) else ""
    return q + " " + a


###############################################################################
# 3) TF–IDF Vectorizer
###############################################################################
def build_tfidf_vectorizer():
    """
    Return a TfidfVectorizer instance.
    Adjust params (ngram_range, max_features, min_df, etc.) to tune performance.
    """
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),  # unigrams + bigrams
        min_df=2,           # ignore terms that appear <2 times
        max_features=10000, # cap features for memory/speed
    )
    return vectorizer


###############################################################################
# 4) Thai BERT Embeddings
###############################################################################
class ThaiBERTEmbedder:
    """
    Use a pretrained Thai BERT or multilingual model from Hugging Face to generate
    sentence embeddings from the [CLS] token (or a pooling strategy).
    """
    def __init__(self, model_name="airesearch/wangchanberta-base-att-spm-uncased", device=None):
        """
        model_name: any Hugging Face model name that suits Thai text, e.g:
                    "airesearch/wangchanberta-base-att-spm-uncased"
        """
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loading BERT tokenizer/model for: {model_name} on device: {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def encode(self, text_list, batch_size=16, max_length=128):
        """
        Convert each text in text_list into a BERT embedding (numpy).
        Returns array shape: (num_texts, hidden_dim).
        """
        all_embs = []
        for i in range(0, len(text_list), batch_size):
            batch_text = text_list[i : i + batch_size]
            inputs = self.tokenizer(
                batch_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
            # Use the [CLS] token from last hidden state as a simple representation
            cls_emb = outputs.last_hidden_state[:, 0, :]  # shape [batch_size, hidden_dim]
            all_embs.append(cls_emb.cpu().numpy())
        return np.concatenate(all_embs, axis=0)


###############################################################################
# 5) Preprocess: TF–IDF + BERT
###############################################################################
def preprocess_data(train_df, test_df, tfidf_vectorizer, bert_embedder):
    """
    1) Create train_df["text"] and test_df["text"] by combining question+answer
    2) Fit TF–IDF on train text, transform train & test => X_tfidf_train, X_tfidf_test
    3) Generate BERT embeddings => X_bert_train, X_bert_test
    4) Horizontal stack => X_train, X_test
    5) Return X_train, y_train, X_test
    """
    # Combine question+answer
    train_df["text"] = train_df.apply(lambda r: combine_text(r["question"], r["answer"]), axis=1)
    test_df["text"]  = test_df.apply(lambda r: combine_text(r["question"], r["answer"]), axis=1)

    # Generate TF–IDF
    print("Fitting TF–IDF on training data...")
    X_tfidf_train = tfidf_vectorizer.fit_transform(train_df["text"].tolist())
    print("TF–IDF dimension (train):", X_tfidf_train.shape)

    print("Transforming test data with TF–IDF...")
    X_tfidf_test = tfidf_vectorizer.transform(test_df["text"].tolist())

    # BERT embeddings
    print("Generating BERT embeddings for train...")
    X_bert_train = bert_embedder.encode(train_df["text"].tolist())
    print("BERT dimension (train):", X_bert_train.shape)

    print("Generating BERT embeddings for test...")
    X_bert_test = bert_embedder.encode(test_df["text"].tolist())

    # We need to convert the TF–IDF sparse matrix to dense to horizontally stack
    # If this is very large, consider a different strategy or dimension reduction.
    # For demonstration, we'll do a .toarray() here (which can be memory-heavy).
    print("Converting TF–IDF to dense arrays (could be memory-heavy for large data).")
    X_tfidf_train_arr = X_tfidf_train.toarray()
    X_tfidf_test_arr  = X_tfidf_test.toarray()

    # Concatenate horizontally
    # shape => [n_samples, tfidf_dim + bert_dim]
    X_train = np.hstack([X_tfidf_train_arr, X_bert_train])
    X_test  = np.hstack([X_tfidf_test_arr,  X_bert_test])

    # Score is numeric
    y_train = train_df["score"].values

    return X_train, y_train, X_test


###############################################################################
# 6) Cross-Validation
###############################################################################
def run_cross_val(X, y, n_splits=5):
    """
    Simple KFold cross-validation to estimate MSE.
    We'll create a new XGBRegressor each fold.
    Adjust hyperparameters or do GridSearchCV for better performance.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    mses = []
    for train_idx, val_idx in kf.split(X):
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        # Basic XGBoost Regressor
        model = xgb.XGBRegressor(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            tree_method="auto",  # 'gpu_hist' if GPU available
        )
        model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], verbose=False)

        y_pred_val = model.predict(X_val_fold)
        fold_mse = mean_squared_error(y_val_fold, y_pred_val)
        mses.append(fold_mse)
    return mses


###############################################################################
# 7) Final Training & Prediction
###############################################################################
def train_final_model(X, y):
    """
    Train a final XGBoost regressor on the full training data.
    Returns the fitted model.
    """
    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="auto",
    )
    model.fit(X, y, verbose=True)
    return model


def predict_and_save(model, X_test, sub_df, output_name="submission.csv"):
    """
    Use the trained model to predict on test set embeddings.
    Insert predictions into sub_df['score'] and save as CSV.
    """
    predictions = model.predict(X_test)
    sub_df["score"] = predictions
    sub_df.to_csv(output_name, index=False)
    print(f"Submission saved to: {output_name}")


###############################################################################
# Main Script
###############################################################################
def main():
    print("=== (1) Reading data ===")
    train_df, test_df, sub_df = read_data()

    print("=== (2) Building TF–IDF vectorizer ===")
    tfidf_vectorizer = build_tfidf_vectorizer()

    print("=== (3) Initializing Thai BERT embedder ===")
    bert_embedder = ThaiBERTEmbedder(
        model_name="airesearch/wangchanberta-base-att-spm-uncased",
        device=None  # auto-detect GPU if available
    )

    print("=== (4) Preprocessing: TF–IDF + BERT embeddings ===")
    X_train, y_train, X_test = preprocess_data(train_df, test_df, tfidf_vectorizer, bert_embedder)

    print("=== (5) Cross-validation ===")
    mses = run_cross_val(X_train, y_train, n_splits=5)
    print("Fold MSEs:", mses)
    print("Mean CV MSE:", np.mean(mses))

    print("=== (6) Train final model on all data ===")
    final_model = train_final_model(X_train, y_train)

    print("=== (7) Predict & Save Submission ===")
    predict_and_save(final_model, X_test, sub_df, output_name="submission.csv")

    print("All done! Check submission.csv for your predictions.")


if __name__ == "__main__":
    main()


=== (1) Reading data ===
=== (2) Building TF–IDF vectorizer ===
=== (3) Initializing Thai BERT embedder ===
Loading BERT tokenizer/model for: airesearch/wangchanberta-base-att-spm-uncased on device: cpu
=== (4) Preprocessing: TF–IDF + BERT embeddings ===
Fitting TF–IDF on training data...
TF–IDF dimension (train): (362, 3193)
Transforming test data with TF–IDF...
Generating BERT embeddings for train...
BERT dimension (train): (362, 768)
Generating BERT embeddings for test...
Converting TF–IDF to dense arrays (could be memory-heavy for large data).
=== (5) Cross-validation ===
Fold MSEs: [np.float64(2.059227445591137), np.float64(2.0891260255989073), np.float64(2.508955236798281), np.float64(2.0387220895365434), np.float64(2.458090508113133)]
Mean CV MSE: 2.2308242611276
=== (6) Train final model on all data ===
=== (7) Predict & Save Submission ===
Submission saved to: submission.csv
All done! Check submission.csv for your predictions.
